# Word Count — Hello World for Distributed Computing

Counts the most frequent words in *Pride and Prejudice* (Project Gutenberg).  
This is the example used in Chapter 01 of the Spark Book to walk through the cluster architecture.

> **Databricks version** — `SparkSession` creation and `spark.stop()` removed (Databricks provides `spark` automatically).  
> Source text is stored in Unity Catalog Volume `learning.chap1.intro`.

## Setup

Create the Unity Catalog hierarchy (`learning` → `chap1` → `intro` volume), then download  
*Pride and Prejudice* from Project Gutenberg into the volume.

In [ ]:
spark.sql("CREATE CATALOG IF NOT EXISTS learning")
spark.sql("CREATE SCHEMA IF NOT EXISTS learning.chap1")
spark.sql("CREATE VOLUME IF NOT EXISTS learning.chap1.intro")
print("Catalog / schema / volume ready.")

In [ ]:
import urllib.request

urllib.request.urlretrieve(
    "https://www.gutenberg.org/files/1342/1342-0.txt",
    "/Volumes/learning/chap1/intro/1342-0.txt",
)
print("Downloaded to /Volumes/learning/chap1/intro/1342-0.txt")

## Read

`spark.read.text()` returns a DataFrame with one column (`value`) and one row per line.  
This call is **lazy** — the file is not read until an action fires.

In [ ]:
import pyspark.sql.functions as F

book = spark.read.text("/Volumes/learning/chap1/intro/1342-0.txt")
book.printSchema()

## Transform

All five steps below are **transformations** — Spark records each instruction and moves on.  
No data moves until the action in the next cell.

In [ ]:
top_words = (
    book
    .select(F.explode(F.split("value", " ")).alias("word"))    # one word per row
    .select(
        F.regexp_extract(F.lower(F.col("word")), "[a-z]+", 0)  # lowercase FIRST, then strip punctuation
         .alias("word")
    )
    .filter(F.col("word") != "")                               # drop empty strings
    .groupBy("word")
    .count()
    .orderBy(F.col("count").desc())
)

## Action

`.show(10)` is the first **action** — this is the moment Spark executes everything above:  
reads the file, splits, cleans, groups, counts, sorts, and returns the top 10 words.

In [ ]:
top_words.show(10)

## Why `show(n)` always shows `isFinalPlan=false`

`show(10)` is implemented as `limit(10).queryExecution` inside the JVM — it builds a *new* Dataset
with a `Limit` operator on top and executes **that** plan. The original `top_words.queryExecution`
is never touched, so its `AdaptiveSparkPlanExec` stays at `isFinalPlan=false`.

To execute through the original `queryExecution`, use `collect()`.
This file is ~700 KB so it is safe here; on large data use `df.write.format("noop").save()` instead.

In [ ]:
_ = top_words.collect()

## Inspect the plan — after AQE finalises

`explain()` on an executed DataFrame shows `isFinalPlan=true` and prints both the
**Final Plan** (what actually ran) and the **Initial Plan** (the pre-execution estimate).

**`ShuffleQueryStage 0` / `1`** — AQE wraps each `Exchange` in a `ShuffleQueryStage`.
After each one completes, AQE reads the actual partition byte counts from `MapOutputTracker`
and re-plans downstream stages before they start.

**`AQEShuffleRead coalesced`** — the initial plan estimated 200 shuffle partitions for both
exchanges. After execution, AQE saw the actual output was tiny and coalesced those 200 partitions
down (likely to 1 each). `coalesced` means partitions were merged; `skewed` would mean a large
partition was split.

**`*(1)`, `*(2)`, `*(3)`** — the `*` prefix means WholeStageCodegen (Tungsten) compiled those
operators into a single JVM function. The number is the codegen stage:

- `*(1)`: Generate → Filter → Project → partial HashAggregate — fused into one loop, runs in Stage 0
- `*(2)`: final HashAggregate — Stage 1, its own compiled function
- `*(3)`: Sort — Stage 2, its own compiled function

`Exchange` and `AQEShuffleRead` carry no `*` — they are shuffle boundaries; data must cross the
network, so it cannot stay in a register-level pipeline.

In [ ]:
top_words.explain()